In [ ]:
using Plots
using Distributed
using LinearAlgebra

num_cores = length(Sys.cpu_info())
if nprocs()==1
    addprocs(num_cores; exeflags=`--project=$(Base.active_project())`)
end

@everywhere begin
    using LatticeAlgorithms
    using LinearAlgebra
    using Dates
    using BlockDiagonals
end
using JLD2

In [ ]:
println("num_cores = $(num_cores)")

In [ ]:
type_lattice = "surface_hexagonal"
χ = 16

# dmin, dmax = 3, 39
# σrange = vcat(0.50:0.01:0.59, 0.591:0.001:0.607, exp(-1/2))

dmin, dmax = 3, 5
σrange = 0.596:0.001:0.607

drange = dmin : 2 : dmax
σdrange = []
for σ in σrange
    for d in drange
        push!(σdrange, [σ, d])
    end
end

num_samples = Int(1e6)


num_samples_each_core = Int(ceil(num_samples/num_cores))
num_total_samples = Int(num_samples_each_core * num_cores);
println([num_samples_each_core, num_samples, num_total_samples])

logfile = "$(type_lattice)_tn_$(drange[1])_$(drange[end])_$(min(σrange...))_$(max(σrange...))_$(χ)_$(num_total_samples)_log.txt"
    open(logfile, "w") do file
end

In [ ]:
@time results = pmap(1:num_cores) do _
    p_list = Dict(σdrange.=>[[0.0, 0.0, 0.0, 0.0] for _ in 1 : length(σdrange)])
    t_list = Dict(σdrange.=>[0.0 for _ in 1 : length(σdrange)])
    f_list = Dict(σdrange.=>[0   for _ in 1 : length(σdrange)])
    truncs = Dict(σdrange.=>[[0.0, 0.0, 0.0, 0.0] for _ in 1 : length(σdrange)])

    for (ind_σd, σd) in enumerate(σdrange)
        σ, d = σd[1], Int(σd[2])
        num_qubits = d^2
        
        TN, indices = tn_template_surf_hex(d)
        
        S_hex = [2 1; 0 √3] / (12)^(1/4);
        S_hex_T = Matrix(transpose(S_hex))
        bigS_T = BlockDiagonal([S_hex_T for _ in 1:num_qubits])
        bigS = BlockDiagonal([S_hex for _ in 1:num_qubits])
        M0 = surface_code_M(d)
        M = M0 * bigS_T; 
        Mperp = GKP_logical_operator_generator(M)            
        Ω = Ω_matrix(M)
        
        X_logical = surface_code_X_logicals(d)[1]
        X = zeros(2num_qubits)
        X[2 .* X_logical .- 1] .= 1
        Z_logical = surface_code_Z_logicals(d)[1]        
        Z = zeros(2num_qubits)
        Z[2 .* Z_logical] .= 1

        Z = bigS * Z .* √π
        X = bigS * X .* √π
        
        pp = [0.0, 0.0, 0.0, 0.0]
        tt = 0.0
        fail = 0
        num_truncs = [0.0, 0.0, 0.0, 0.0]
        σdtime = @elapsed for _ in 1 : num_samples_each_core
            ξ = σ * randn(2num_qubits)
            
            if d <=17
                setprecision(BigFloat, 64)
                ξ = BigFloat.(ξ)                    
            end
            
            ξ2 = -√(2π) * M * Ω * ξ
            s = ξ2 - floor.(ξ2/(2π)) * 2π
            ηs = -transpose(Ω*Mperp) * s/√(2π) ; 
            b = inv(√(2π) * transpose(Mperp)) * (ηs-ξ)
            # @assert norm(round.(Int, b) - b) < 1e-10
            
            tt += @elapsed begin
                lstar = nothing
                try
                    lstar, prob_I, prob_X, prob_Y, prob_Z, num_truncs_I, num_truncs_X, num_truncs_Y, num_truncs_Z = tn_surf_hex(ηs, σ, TN, indices, Z, X, χ)
                    num_truncs[1] += num_truncs_I
                    num_truncs[2] += num_truncs_X
                    num_truncs[3] += num_truncs_Y
                    num_truncs[4] += num_truncs_Z
                catch
                    fail += 1
                end

                lstar == nothing && (lstar = zeros(length(ηs)))
                
                rec = -ηs + lstar
                final_error = (rec+ξ)

                nx = transpose(final_error) * Ω * Z / π
                nz = transpose(final_error) * Ω * X / π
                nx = mod(round(nx), 2)
                nz = mod(round(nz), 2)
                @assert round.(Int, nx) ≈ nx
                @assert round.(Int, nz) ≈ nz  

                ##########
                if nx ≈ 0 && nz ≈ 0
                    pp[1] += 1
                elseif nx ≈ 1 && nz ≈ 0
                    pp[2] += 1
                elseif nx ≈ 0 && nz ≈ 1
                    pp[3] += 1
                elseif nx ≈ 1 && nz ≈ 1
                    pp[4] += 1
                end                    
            end
        end

        p_list[[σ, d]] .+= pp
        t_list[[σ, d]] += tt
        f_list[[σ, d]] += fail
        truncs[[σ, d]] .+= num_truncs

        if myid() == 2
            # Print the progress of the 2nd worker
            println(["$(ind_σd)/$(length(σdrange)), $d, $(σdtime), $(string(now()))"])
            open(logfile, "a") do file
                write(file, "$(ind_σd)/$(length(σdrange)), $d, $(σdtime), $(string(now()))\n")
            end
        end
    end
    return (p_list, t_list, f_list, truncs)
end ;

p_list = merge(.+, [res[1] for res in results]...)
t_list = merge(+, [res[2] for res in results]...)
f_list_0 = merge(+, [res[3] for res in results]...)
truncs_list = merge(.+, [res[4] for res in results]...)

f_list = Dict()
for (k, v) in f_list_0
    f_list[k] = Float64(v ./ num_total_samples)
end

map!(v->v./num_total_samples, values(t_list))
map!(v->v./num_total_samples, values(p_list))
map!(v->v./num_total_samples, values(truncs_list))

c_list = Dict()

for (k, v) in p_list
    c_list[k] = coherent_information_pauli_channel(v[2], v[4], v[3])
end



# Save the result
fn = "$(type_lattice)_tn_$(drange[1])_$(drange[end])_$(min(σrange...))_$(max(σrange...))_$(χ)_$(num_total_samples).jld2"
jldsave(fn; 
    σrange=σrange, 
    num_samples=num_samples_each_core*num_cores,
    p_list = p_list,
    t_list = t_list,
    c_list = c_list,
    f_list = f_list,
    trunc_list = truncs_list,
    drange = drange
)


In [ ]:
sort(load(fn))

# Compare to existing data

In [ ]:
function get_p0list_sorted(p_list, drange, σrange)
    p0list_sorted = sort(p_list)
    p0list_sorted = collect(values(p0list_sorted))
    p0list_sorted = reshape(p0list_sorted, (length(drange), length(σrange)))
    p0list_sorted = [p0list_sorted[:,i] for i in 1:size(p0list_sorted,2)]
    return p0list_sorted
end


In [ ]:
new_data = sort(load(fn))
new_p_list = new_data["p_list"]
new_p_list_sorted = get_p0list_sorted(new_p_list, drange, σrange)

In [ ]:
## 
old_data = sort(load("data/surface_hexagonal_tn_3_15_0.596_0.607_16_1023680.jld2"))
old_p_list = old_data["p_list"]

old_p_list_d = Dict()
for (k, v) in old_p_list
    if drange[1] <= k[2] <= drange[end]
        old_p_list_d[k] = v
    end
end

old_p_list_sorted = get_p0list_sorted(old_p_list_d, drange, old_data["σrange"])
display(old_p_list_sorted)


In [ ]:
gs = []
linecolors = get_color_palette(:auto, plot_color(:white))

for i in 1:4
    g = plot()
    for (dind, d) in enumerate(drange)
        new_p_list = [item[dind][i] for item in new_p_list_sorted]
        old_p_list = [item[dind][i] for item in old_p_list_sorted]
        new_yerr = sqrt.(new_p_list .* (1 .- new_p_list) ./ num_total_samples)
        old_yerr = sqrt.(old_p_list .* (1 .- old_p_list) ./ num_total_samples)
        plot!(σrange, new_p_list, marker=:circle, label="new data, i=$i, d=$d", linecolor=linecolors[dind], yerr=new_yerr)
        plot!(old_data["σrange"], old_p_list, marker=:star, label="old data 1, i=$i, d=$d", linecolor=linecolors[dind], yerr=old_yerr)
    end
    push!(gs, g)    
end
plot(gs..., size=(1000, 1000))